# Anonymize data

* replace original panel ids with new ids
* original output
* ls tasks
* data derrived from ls tasks 



In [102]:
import json
import pandas as pd
import os
import csv

## Step 1: Create mapping including all ids

In [52]:
df_original = pd.read_excel('../data/final_dataset.xlsx')

In [53]:
df_original

,Panel,StartDate,EndDate,Status,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,DistributionChannel,...,Reject: did not finish the study,Reject: Geen consent,Reject: attentioncheck wrong,Reject: less than 20 words,Reject: misunderstood assignment,ChatGPT suspission: punctuation,ChatGPT suspission: same structure across four texts,ChatGPT suspission: fast completion time (too high WPM),Reject: timelimit,WPM
0,Panel,Start Date,End Date,Response Type,Progress,Duration (in seconds),Finished,Recorded Date,Response ID,Distribution Channel,...,Reject: did not finish the study,Reject: Geen consent,Reject: attentioncheck wrong,Reject: less than 20 words,Reject: misunderstood assignment,"ChatGPT suspission: punctiation, emoijs",ChatGPT suspission: same structure across four...,ChatGPT suspission: fast completion time (too ...,Reject: timelimit,WPM
1,Panel Inzicht,2025-07-07 04:03:05,2025-07-07 04:05:00,IP Address,100,115,True,2025-07-07 04:05:01.386000,R_8hhieTlGe8ymTfs,anonymous,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN
2,Panel Inzicht,2025-05-28 02:57:11,2025-05-28 02:59:27,IP Address,100,136,True,2025-05-28 02:59:27.931000,R_2OPvv5utG60eyAM,anonymous,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN
3,Panel Inzicht,2025-07-01 02:26:31,2025-07-01 02:29:04,IP Address,100,152,True,2025-07-01 02:29:04.630000,R_2CrQHnid4AiFLNw,anonymous,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN
4,Panel Inzicht,2025-05-23 02:00:43,2025-05-23 02:03:38,IP Address,100,175,True,2025-05-23 02:03:39.099000,R_2dWCJfxF5BaI4vV,anonymous,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
871,Panel Inzicht,2025-07-16 04:20:53,2025-07-16 04:24:32,IP Address,100,219,True,2025-07-16 04:24:33.337000,R_210zHfSHqJzttUs,anonymous,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
872,Panel Inzicht,2025-07-09 06:03:50,2025-07-09 06:15:30,IP Address,86,699,False,2025-07-16 06:03:51.622000,R_8PoQGxWe93LXE7M,anonymous,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
873,Panel Inzicht,2025-07-16 10:06:49,2025-07-16 10:07:06,IP Address,100,16,True,2025-07-16 10:07:06.407000,R_2f6yKrMp60BYnMB,anonymous,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
874,Panel Inzicht,2025-07-09 10:34:50,2025-07-09 10:36:27,IP Address,86,97,False,2025-07-16 10:34:52.901000,R_8norphtOEJ0llzt,anonymous,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
header = df_original.columns

In [56]:
'ParticipantID' in header

True

In [57]:
# Get all participant ids

part_id = df_original['ParticipantID'][1:]
set_part_id = set(part_id)

In [58]:
print(len(part_id))
print(len(set_part_id))

875
800


In [59]:
# make mapping

mapping = dict()
orig_new_dict = dict()


for n, pid in enumerate(set_part_id):
    mapping[n] = pid
    orig_new_dict[pid] = n
 
with open('../data/IDmapping.json', 'w') as outfile:
    json.dump(mapping, outfile)

## Step 2: Replace ids in original excel sheet

In [71]:
# Make new dir

new_dir = '../dataANON'

if not os.path.exists(new_dir):
    os.mkdir(new_dir)

In [60]:
# replace in original file

for i, row in df_original.iterrows():
    pid = row['ParticipantID']
    if i > 0:
        new_id = orig_new_dict[pid]
        row['ParticipantID'] = new_id

In [61]:
df_original.to_excel('../dataANON/final_dataset_anonymized.xlsx')

/var/folders/y9/gpzq16dd7sd129brk7fnpzn00000gn/T/ipykernel_58474/354946285.py:1: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  df_original.to_excel('../dataANON/final_dataset_anonymized.xlsx')


## Step 3: Replace ids in all ls input and output files

In [116]:
names = [
    'kim-full-data',
    'kim-full-data-update1',
    'AnnaIAABehav',
    'KimIAABehav',
    'MerelIAARef1',
    'MerelIAARef2',
    'Annotations-jan',]
    

In [117]:
for name in names:  
    print(name)
    path = f'../data/{name}.json'
    anon_path = f'../dataANON/{name}.json'
    with open(path) as infile:
        data = json.load(infile)

    for d in data:
        #print(d['data']['text'])
        text = d['data']['text']
        text_id, text = text.split('\nText: ')
        pid, cond = text_id.split(' ', 1)
        new_id = orig_new_dict[pid]
        new_text = f'{new_id} {cond}\nText: {text}'
        d['data']['text'] = new_text

    json_str = json.dumps(data)

    with open(anon_path, 'w') as outfile:
        outfile.write(json_str)

kim-full-data
kim-full-data-update1
AnnaIAABehav
KimIAABehav
MerelIAARef1
MerelIAARef2
Annotations-jan


## Step 4: Replace in results

In [118]:
# Make new dir

new_dir = '../resultsANON'

if not os.path.exists(new_dir):
    os.mkdir(new_dir)

In [119]:
names = [
        'AnnaIAABehav',
        'KimIAABehav',
        'Annotations-jan',
        'kim-full-data',
        'kim-full-data-update1',
]

In [120]:
# Get all filenames

# Replace in all filenames

for name in names:
    path = f'../results/{name}/'
    new_path = f'../resultsANON/{name}/'
    if not os.path.exists(new_path):
        os.mkdir(new_path)

    for f in os.listdir(path):
        if f != '.ipynb_checkpoints':
            # print(f)
            old_full_path = f'{path}{f}'

            pid, cond = f.split(' ', 1)
            pid = pid.split('output-')[1]
            new_id = orig_new_dict[pid]
            new_f = f'output-{new_id} {cond}'
            new_full_path = f'../resultsANON/{name}/{new_f}'

            with open(old_full_path) as infile:
                content = infile.read()
            with open(new_full_path, 'w') as outfile:
                outfile.write(content)

## Step 4: Replace in results overviews

In [94]:
# Results overviews

names = ['Annotations-jan', 'kim-full-data-update1']

for name in names:
    path = f'../results/{name}.xlsx'
    path_new = f'../resultsANON/{name}.xlsx'

    df_overview = pd.read_excel(path)

    for i, row in df_overview.iterrows():
        pid = row['Article_id'].split('output-')[1]
        new_id = orig_new_dict[pid]
        row['Aricle_id'] = f'output-{new_id}'

    df_overview.to_excel(path_new)

/var/folders/y9/gpzq16dd7sd129brk7fnpzn00000gn/T/ipykernel_58474/1960097794.py:16: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  df_overview.to_excel(path_new)
/var/folders/y9/gpzq16dd7sd129brk7fnpzn00000gn/T/ipykernel_58474/1960097794.py:16: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  df_overview.to_excel(path_new)


## Step 5: Replace in variables

In [95]:
new_dir = '../variablesANON'

if not os.path.exists(new_dir):
    os.mkdir(new_dir)

In [109]:
# get all current variables:

old_dir = '../variables/'
new_dir = '../variablesANON/'

for f in os.listdir(old_dir):
    print(f)
    if f.endswith('.csv'):
        with open(f'{old_dir}{f}') as infile:
            data = list(csv.DictReader(infile, delimiter=','))
            
        for d in data:
            pid = d['Article_id'].split('output-')[1]
            new_id = orig_new_dict[pid]
            d['Article_id'] = f'output-{new_id}'
        
        header = data[0].keys()
        with open(f'{new_dir}{f}', 'w') as outfile:
            writer = csv.DictWriter(outfile, fieldnames = header, delimiter = ',')
            writer.writeheader()
            for d in data:
                writer.writerow(d)

negated-references-presence-proportion.csv
generalization-negated-references.csv
opposite-references-proportion.csv
text-overview.csv
generalization-references.csv
n_words_ref_100.csv
generalization-proportion-references.csv
n_words_text.csv
generalization-proportion-references-negated.csv
mean_words_ref_text.csv
.ipynb_checkpoints
total-references.csv
opposite-references-present.csv
